<a href="https://colab.research.google.com/github/Wilmot2025/Java_Test/blob/main/medicine_price_predication_85_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
miadul_medicine_price_predication_dataset_path = kagglehub.dataset_download('miadul/medicine-price-predication-dataset')

print('Data source import complete.')


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ML models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import AdaBoostRegressor
from sklearn.linear_model import Ridge, Lasso

import joblib


import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))



In [ ]:
df = pd.read_csv("/kaggle/input/medicine-price-predication-dataset/medicine_price_dataset.csv")
df.head()


In [ ]:
df.shape, df.info()


In [ ]:
df.describe()


In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df["price"], bins=30, kde=True)
plt.title("Price Distribution")
plt.show()


In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x="category", data=df)
plt.title("Medicine Category Distribution")
plt.show()


In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x="demand_level", y="price", data=df)
plt.title("Demand Level vs Price")
plt.show()


In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()


In [ ]:
label_encoders = {}

categorical_cols = ["medicine_name", "category", "company", "import_status", "demand_level"]

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le


In [ ]:
X = df.drop(columns=["price"])
y = df["price"]


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "KNN": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest": RandomForestRegressor(),
    "Gradient Boosting": GradientBoostingRegressor(),
    "AdaBoost": AdaBoostRegressor(),
    "SVR": SVR(),
}


In [ ]:
results = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)

    mae = mean_absolute_error(y_test, preds)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, preds)

    results.append([name, mae, rmse, r2])

results_df = pd.DataFrame(results, columns=["Model", "MAE", "RMSE", "R2 Score"])
results_df.sort_values(by="R2 Score", ascending=False)


In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(x="R2 Score", y="Model", data=results_df)
plt.title("Model Performance Comparison")
plt.show()


In [ ]:
rf = RandomForestRegressor()
rf.fit(X_train_scaled, y_train)

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
}).sort_values(by="Importance", ascending=False)

feature_importance


In [ ]:
sample = X_test.iloc[[4]]
predicted_price = rf.predict(scaler.transform(sample))

print("Predicted Price:", predicted_price)
